In [1]:
import pandas as pd

import yaml
import os
from src.utils import get_project_root, load_config
from typing import Tuple, Any, Dict

In [2]:
config = load_config()

In [3]:
config

{'path_raw_data': 'data/raw/credit_risk_dataset.csv',
 'target_col': 'loan_status',
 'random_state': 123,
 'path_train_set': ['data/interim/X_train.pkl', 'data/interim/y_train.pkl'],
 'path_valid_set': ['data/interim/X_valid.pkl', 'data/interim/y_valid.pkl'],
 'path_test_set': ['data/interim/X_test.pkl', 'data/interim/y_test.pkl'],
 'path_imputer_median': 'models/imputer_median.pkl',
 'path_train_clean': ['data/processed/X_train_clean.pkl',
  'data/processed/y_train_clean.pkl'],
 'path_valid_clean': ['data/processed/X_valid_clean.pkl',
  'data/processed/y_valid_clean.pkl'],
 'path_test_clean': ['data/processed/X_test_clean.pkl',
  'data/processed/y_test_clean.pkl']}

In [4]:
def save_to_config(key: str, value: any, filename: str = "config.yaml"):
    """
    Menyimpan key dan value baru LANGSUNG ke file config.yaml
    tanpa merusak path yang sudah ada.
    """
    path_config = get_project_root() / "config" / filename
    
    with open(path_config, "r") as file:
        raw_config = yaml.safe_load(file) or {}
        
    raw_config[key] = value
    
    with open(path_config, "w") as file:
        yaml.dump(raw_config, file, default_flow_style=False, sort_keys=True)
        
    print(f"Berhasil menyimpan permanen: '{key}' ke {filename}")

In [5]:
def load_data(fname: str) -> pd.DataFrame:
    """
    Memuat dataset dari file csv ke dalam pandas dataframe.

    Args:
        fname (str): lokasi file (path) .csv 

    Returns: 
        pd.DataFrame: DataFrame yang berisi data dari file CSV tersebut
    """
    root = get_project_root()
    filepath  = root / fname
    
    if not filepath.exists():
        raise FileNotFoundError(f"File tidak ditemukan di {filepath}")
    
    data = pd.read_csv(filepath)
    print(f"Data Shape: {data.shape}")

    return data

In [6]:
PATH_RAW_DATA = "data/raw/credit_risk_dataset.csv"

In [7]:
save_to_config("path_raw_data", PATH_RAW_DATA)

Berhasil menyimpan permanen: 'path_raw_data' ke config.yaml


In [8]:
FNAME = config["path_raw_data"]
data = load_data(FNAME)

Data Shape: (32581, 12)


In [9]:
data.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


In [10]:
def split_input_output(data: pd.DataFrame, target_col: str) -> Tuple[pd.DataFrame, pd.Series]:
    """
    Memisahkan dataset menjadi fitur (X) dan target (y)

    Fungsi ini menghapus kolom target dari dataframe utama untuk membuat
    kumpulan fitur, dan mengekstrak kolom target tersebut secara terpisah 

    Args:
        data (pd.DataFrame): Dataframe asli (raw) yang berisi fitur dan target
        target_col (str): Nama kolom yang akan dijadikan label/target

    Returns:
        Tuple[pd.DataFrame, pd.Series]:
            - X (pd.DataFrame): fitur (semua kolom kecuali target)
            - y (pd.Series): target (hanya kolom target)
    """

    X = data.drop(target_col, axis=1)
    y = data[target_col]

    print(f"Original data shape: {data.shape}")
    print(f"X data shape       : {X.shape}")
    print(f"y data shape       : {y.shape}")

    return X, y

In [11]:
save_to_config("target_col", "loan_status")

Berhasil menyimpan permanen: 'target_col' ke config.yaml


In [12]:
config = load_config()

In [13]:
config

{'path_imputer_median': 'models/imputer_median.pkl',
 'path_raw_data': 'data/raw/credit_risk_dataset.csv',
 'path_test_clean': ['data/processed/X_test_clean.pkl',
  'data/processed/y_test_clean.pkl'],
 'path_test_set': ['data/interim/X_test.pkl', 'data/interim/y_test.pkl'],
 'path_train_clean': ['data/processed/X_train_clean.pkl',
  'data/processed/y_train_clean.pkl'],
 'path_train_set': ['data/interim/X_train.pkl', 'data/interim/y_train.pkl'],
 'path_valid_clean': ['data/processed/X_valid_clean.pkl',
  'data/processed/y_valid_clean.pkl'],
 'path_valid_set': ['data/interim/X_valid.pkl', 'data/interim/y_valid.pkl'],
 'random_state': 123,
 'target_col': 'loan_status'}

In [14]:
TARGET_COL = config["target_col"]
X, y = split_input_output(data, TARGET_COL)

Original data shape: (32581, 12)
X data shape       : (32581, 11)
y data shape       : (32581,)


In [15]:
X.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,0.55,Y,4


In [16]:
y.head()

0    1
1    0
2    1
3    1
4    1
Name: loan_status, dtype: int64

In [17]:
from sklearn.model_selection import train_test_split

In [18]:
def split_train_test(
        X: pd.DataFrame,
        y: pd.Series,
        test_size: float,
        random_state: int
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """
    Membagi data menjadi training set dan testing set
    Fungsi ini menggunakan train_test_split dari sklearn untuk memastikan 
    konsistensi random state

    Args:
        X (pd.DataFrame): Fitur
        y (pd.Series): Target
        test_size (float): Proporsi data untuk test set (0.0 - 1.0)
        random_state (int): Seed untuk reproduksibilitas hasil

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
            Urutannya adalah: (X_train, X_test, y_train, y_test)
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    print(f"X train shape: {X_train.shape}")
    print(f"X test shape : {X_test.shape}")
    print(f"y train shape: {y_train.shape}")
    print(f"y test shape : {y_test.shape}")

    return X_train, X_test, y_train, y_test

In [19]:
save_to_config("random_state", 123)

Berhasil menyimpan permanen: 'random_state' ke config.yaml


In [20]:
config = load_config()

In [21]:
X_train, X_non_train, y_train, y_non_train = split_train_test(
    X, y,
    test_size=0.2,
    random_state=config["random_state"]
)

X_valid, X_test, y_valid, y_test = split_train_test(
    X=X_non_train,
    y=y_non_train,
    test_size=0.5,
    random_state=config["random_state"]
)

X train shape: (26064, 11)
X test shape : (6517, 11)
y train shape: (26064,)
y test shape : (6517,)
X train shape: (3258, 11)
X test shape : (3259, 11)
y train shape: (3258,)
y test shape : (3259,)


In [22]:
import joblib

In [23]:
def serialize_data(data: Any, path: str) -> None:
    """
    Menyimpan python object ke dalam file serealisasi 
    Fungsi ini menggunakan library joblib untuk mengubah objek seperti
    DataFrame atau Model menjadi file .pkl agar bisa digunakan kembali
    tanpa harus memproses ulang dari awal.

    Args:
        data (Any): Objek yang ingin disimpan, bisa berupa DataFrame, list, atau model
        path (str): Lokasi dan nama file tujuan
    """
    root = get_project_root()
    filepath  = root / path

    os.makedirs(filepath.parent, exist_ok=True)
    
    joblib.dump(data, filepath)

    print(f"-> Sukses menyimpan data ke: {filepath}")


In [24]:
save_to_config(
    key="path_train_set",
    value=["data/interim/X_train.pkl", "data/interim/y_train.pkl"]
)

Berhasil menyimpan permanen: 'path_train_set' ke config.yaml


In [25]:
save_to_config(
    key="path_valid_set",
    value=["data/interim/X_valid.pkl", "data/interim/y_valid.pkl"]
)

Berhasil menyimpan permanen: 'path_valid_set' ke config.yaml


In [26]:
save_to_config(
    key="path_test_set",
    value=["data/interim/X_test.pkl", "data/interim/y_test.pkl"]
)

Berhasil menyimpan permanen: 'path_test_set' ke config.yaml


In [27]:
config = load_config()

In [28]:
path_X_train, path_y_train = config["path_train_set"]
path_X_valid, path_y_valid = config["path_valid_set"]
path_X_test, path_y_test = config["path_test_set"]

In [29]:
serialize_data(X_train, path_X_train)
serialize_data(y_train, path_y_train)

serialize_data(X_valid, path_X_valid)
serialize_data(y_valid, path_y_valid)

serialize_data(X_test, path_X_test)
serialize_data(y_test, path_y_test)

-> Sukses menyimpan data ke: /home/amadeo/Pacmann/MLProcess/Mentoring/AMADEO_MLPROCESS/data/interim/X_train.pkl
-> Sukses menyimpan data ke: /home/amadeo/Pacmann/MLProcess/Mentoring/AMADEO_MLPROCESS/data/interim/y_train.pkl
-> Sukses menyimpan data ke: /home/amadeo/Pacmann/MLProcess/Mentoring/AMADEO_MLPROCESS/data/interim/X_valid.pkl
-> Sukses menyimpan data ke: /home/amadeo/Pacmann/MLProcess/Mentoring/AMADEO_MLPROCESS/data/interim/y_valid.pkl
-> Sukses menyimpan data ke: /home/amadeo/Pacmann/MLProcess/Mentoring/AMADEO_MLPROCESS/data/interim/X_test.pkl
-> Sukses menyimpan data ke: /home/amadeo/Pacmann/MLProcess/Mentoring/AMADEO_MLPROCESS/data/interim/y_test.pkl


In [30]:
def deserialize_data(path: str) -> Any:
    """
    Memuat kembali objek yang telah disimpan dari hasil serealisasi
    Fungsi ini membaca file .pkl yang ada di folder dan mengembalikannya
    ke dalam variabel python

    Args:
        path (str): Lokasi file .pkl yang ingin dimuat

    Returns:
        Any: Objek asli yang sebelumnya disimpan 
    """
    root = get_project_root()
    filepath  = root / path

    if not filepath.exists():
        raise FileNotFoundError(f"File .pkl tidak ditemukan di {filepath}")

    return joblib.load(filepath)


In [31]:
X_coba = deserialize_data(path=path_X_train)

In [32]:
X_coba.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
29762,45,37500,MORTGAGE,1.0,DEBTCONSOLIDATION,B,5000,11.49,0.13,N,16
2714,25,50000,RENT,5.0,PERSONAL,A,12000,7.88,0.24,N,2
50,24,78000,RENT,4.0,DEBTCONSOLIDATION,D,30000,NaN,0.38,Y,4
28458,31,78504,RENT,2.0,EDUCATION,C,10000,11.41,0.13,N,7
3674,26,14000,RENT,2.0,VENTURE,B,4000,NaN,0.29,N,3
